In [1]:
# ============================================================
# SWEEP OF w THRESHOLDS ON THE NEW G_rizoma
# Using the previously applied local semantic filtering
# ============================================================

import networkx as nx
import pandas as pd

# Ensure that G_rizoma already exists
assert "G_rizoma" in globals(), "You must first execute the block that constructs G_rizoma."

# Thresholds to evaluate
W_LIST = [1, 3, 5, 8, 10, 20, 30]

def build_backbone(G, wmin):
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from([
        (u, v, d)
        for u, v, d in G.edges(data=True)
        if float(d.get("weight", 1)) >= wmin
    ])
    H.remove_nodes_from([n for n in list(H.nodes()) if H.degree(n) == 0])
    return H

rows = []

for w in W_LIST:
    H_w = build_backbone(G_rizoma, w)

    if H_w.number_of_nodes() > 0:
        comps = list(nx.connected_components(H_w))
        lcc_nodes = max(comps, key=len)
        L_w = H_w.subgraph(lcc_nodes).copy()

        lcc_prop = L_w.number_of_nodes() / H_w.number_of_nodes()
        density = nx.density(H_w)
        density_lcc = nx.density(L_w)

    else:
        comps = []
        L_w = nx.Graph()
        lcc_prop = 0
        density = 0
        density_lcc = 0

    rows.append({
        "w": w,
        "nodes_graph": H_w.number_of_nodes(),
        "edges_graph": H_w.number_of_edges(),
        "components": len(comps),
        "nodes_LCC": L_w.number_of_nodes(),
        "edges_LCC": L_w.number_of_edges(),
        "LCC_prop": round(lcc_prop, 4),
        "density_graph": density,
        "density_LCC": density_lcc
    })

df_w_sweep = pd.DataFrame(rows)

print("\n==============================")
print("w THRESHOLD SWEEP")
print("==============================")
print(df_w_sweep.to_string(index=False))

# Save results
OUT_CSV = "outputs/w_threshold_sweep.csv"
df_w_sweep.to_csv(OUT_CSV, index=False)

print(f"\nResults saved to: {OUT_CSV}")


w THRESHOLD SWEEP
 w  nodes_graph  edges_graph  components  nodes_LCC  edges_LCC  LCC_prop  density_graph  density_LCC
 1        38618       109022        1839      34316     106297    0.8886       0.000146     0.000181
 3         3063         7214          36       2991       7176    0.9765       0.001538     0.001605
 5         1627         3331          19       1588       3309    0.9760       0.002518     0.002626
 8         1011         1801          11        990       1790    0.9792       0.003528     0.003656
10          783         1344           8        769       1337    0.9821       0.004390     0.004528
20          408          608           2        406        607    0.9951       0.007323     0.007383
30          258          368           2        256        367    0.9922       0.011100     0.011244

Results saved to: outputs/w_threshold_sweep.csv
